In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import joblib


In [40]:
class DatasetDF:
    def __init__(self, excel_path, targets):
        self.df = pd.read_excel(excel_path)
        self.targets = targets

    def split_train_test(self, test_size=0.2, random_state=42):
        self.X = self.df.drop(columns=self.targets + ["Player"])
        self.y = self.df[self.targets]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )
        return self

    def get_player_last_row(self, player_name):
        df_player = self.df[self.df["Player"] == player_name]
        if df_player.empty:
            raise ValueError(f"Joueur '{player_name}' introuvable")
        return df_player.iloc[-1:]

    def available_test_players(self):
        return sorted(self.df.loc[self.X_test.index, "Player"].unique())


In [41]:
class ModelDF:
    def __init__(self, model):
        self.model = model
        self.scaler = StandardScaler()

    def train(self, X_train, y_train):
        X_scaled = self.scaler.fit_transform(X_train)
        self.model.fit(X_scaled, y_train)

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def evaluate(self, X_test, y_test):
        preds = self.predict(X_test)
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "R2": r2_score(y_test, preds)
        }


In [42]:
class ModelTrainerDF:
    def __init__(self, dataset):
        self.dataset = dataset
        self.models = {}
        self.scores = []

    def model_defs(self):
        return {
            "LinearRegression": LinearRegression(),
            "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "SVR": SVR()
        }

    def train(self):
        for target in self.dataset.targets:
            self.models[target] = {}

            for name, base_model in self.model_defs().items():
                model = ModelDF(base_model)
                model.train(self.dataset.X_train, self.dataset.y_train[target])

                metrics = model.evaluate(
                    self.dataset.X_test,
                    self.dataset.y_test[target]
                )

                self.models[target][name] = model

                self.scores.append({
                    "Target": target,
                    "Model": name,
                    "MAE": metrics["MAE"],
                    "R2": metrics["R2"]
                })

        return pd.DataFrame(self.scores).sort_values(["Target", "MAE"])


In [43]:
class PlayerPredictorDF:
    def __init__(self, dataset, trainer):
        self.dataset = dataset
        self.trainer = trainer

    def predict_player(self, player_name):
        row = self.dataset.get_player_last_row(player_name)
        X = row.drop(columns=self.dataset.targets + ["Player"])

        results = {}
        for target, models in self.trainer.models.items():
            results[target] = {}
            for model_name, model in models.items():
                results[target][model_name] = round(float(model.predict(X)[0]), 3)

        return results


In [44]:
def compare_real_vs_predicted_df(dataset, predictor, player_name):
    last_row = dataset.get_player_last_row(player_name)
    real = last_row[dataset.targets].iloc[0]

    preds = predictor.predict_player(player_name)

    rows = []
    for target in dataset.targets:
        for model, value in preds[target].items():
            rows.append({
                "Target": target,
                "Model": model,
                "Real": round(real[target], 3),
                "Predicted": value,
                "Error": round(value - real[target], 3)
            })

    return pd.DataFrame(rows)


In [45]:
EXCEL_PATH_DF = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\data\selection\features_DF_selected.xlsx"

TARGETS_DF = [
    "Tkl/90",
    "Int/90",
    "Blocks/90",
    "Clr/90",
    "Gls/90"
]


In [46]:
dataset_df = DatasetDF(EXCEL_PATH_DF, TARGETS_DF)
dataset_df.split_train_test()

trainer_df = ModelTrainerDF(dataset_df)
results_df = trainer_df.train()

results_df


,Target,Model,MAE,R2
11,Blocks/90,SVR,0.137817,0.787380
10,Blocks/90,GradientBoosting,0.144748,0.754969
8,Blocks/90,LinearRegression,0.147962,0.742105
9,Blocks/90,RandomForest,0.152738,0.729022
14,Clr/90,GradientBoosting,0.346122,0.900804
13,Clr/90,RandomForest,0.366938,0.887146
15,Clr/90,SVR,0.405005,0.858435
12,Clr/90,LinearRegression,0.477174,0.815585
17,Gls/90,RandomForest,0.006658,0.950271
18,Gls/90,GradientBoosting,0.006844,0.958230


In [47]:
dataset_df.available_test_players()[:20]


['Aaron Hickey',
 'Abakar Sylla',
 'Abdel Abqar',
 'Abdul Mumin',
 'Achraf Hakimi',
 'Adam Dźwigała',
 'Adam Masina',
 'Adrien Tameze',
 'Adrien Truffert',
 'Aitor Paredes',
 'Aitor Ruibal',
 'Alejandro Balde',
 'Alejandro Francés',
 'Alessandro Bastoni',
 'Alessandro Vogliacco',
 'Alessio Romagnoli',
 'Alex Telles',
 'Alfonso Espino',
 'Ali Abdi',
 'Andoni Gorosabel']

In [48]:
predictor_df = PlayerPredictorDF(dataset_df, trainer_df)

player_name = dataset_df.available_test_players()[5]
player_name
pd.DataFrame(predictor_df.predict_player(player_name)).T
compare_real_vs_predicted_df(dataset_df, predictor_df, player_name)



,Target,Model,Real,Predicted,Error
0,Tkl/90,LinearRegression,4.878,2.187,-2.691
1,Tkl/90,RandomForest,4.878,2.407,-2.471
2,Tkl/90,GradientBoosting,4.878,2.488,-2.390
3,Tkl/90,SVR,4.878,2.552,-2.326
4,Int/90,LinearRegression,0.976,0.921,-0.055
5,Int/90,RandomForest,0.976,0.517,-0.459
6,Int/90,GradientBoosting,0.976,0.689,-0.287
7,Int/90,SVR,0.976,0.725,-0.251
8,Blocks/90,LinearRegression,1.463,1.137,-0.326
9,Blocks/90,RandomForest,1.463,1.115,-0.348


In [49]:
def top_10_df_all_targets(dataset, trainer):
    rows = []

    for target, models in trainer.models.items():
        best_model = min(
            models,
            key=lambda m: mean_absolute_error(
                dataset.y_test[target],
                models[m].predict(dataset.X_test)
            )
        )

        model = models[best_model]
        preds = model.predict(dataset.X)

        df_tmp = pd.DataFrame({
            "Player": dataset.df["Player"],
            "Predicted": preds
        }).sort_values("Predicted", ascending=False).head(10)

        df_tmp["Target"] = target
        df_tmp["Model"] = best_model

        rows.append(df_tmp)

    return pd.concat(rows)
top_10_df_all_targets(dataset_df, trainer_df)



,Player,Predicted,Target,Model
1797,Lars Ritzka,3.561515,Tkl/90,SVR
1928,Mats Wieffer,3.467878,Tkl/90,SVR
1022,José Luis Gayà,3.467677,Tkl/90,SVR
280,Jonas Hector,3.466606,Tkl/90,SVR
1196,Juanlu Sánchez,3.422339,Tkl/90,SVR
54,Melvin Bard,3.416344,Tkl/90,SVR
1700,Noussair Mazraoui,3.413927,Tkl/90,SVR
1895,Destiny Udogie,3.404809,Tkl/90,SVR
1352,Jon Aramburu,3.351399,Tkl/90,SVR
466,Lucas Oliveira Rosa,3.348884,Tkl/90,SVR


In [53]:
from sklearn.metrics import mean_absolute_error
import joblib
import os

SAVE_DIR = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\ML_Notebooks"
os.makedirs(SAVE_DIR, exist_ok=True)

for target, models in trainer_df.models.items():
    best_model_name = min(
        models,
        key=lambda m: mean_absolute_error(
            dataset_df.y_test[target],
            models[m].predict(dataset_df.X_test)
        )
    )

    best_model = models[best_model_name]

    joblib.dump(
        {
            "model_name": best_model_name,
            "scaler": best_model.scaler,
            "model": best_model.model
        },
        os.path.join(
            SAVE_DIR,
            f"best_DF_{target.replace('/', '_').replace(' ', '_')}.pkl"
        )
    )

    print(f"✅ Modèle sauvegardé : best_DF_{target}")


✅ Modèle sauvegardé : best_DF_Tkl/90
✅ Modèle sauvegardé : best_DF_Int/90
✅ Modèle sauvegardé : best_DF_Blocks/90
✅ Modèle sauvegardé : best_DF_Clr/90
✅ Modèle sauvegardé : best_DF_Gls/90
